In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/ecommerce-etl-pipeline'
PROCESSED = f'{BASE}/data/processed'

# Load all cleaned tables
orders      = pd.read_csv(f'{PROCESSED}/orders_clean.csv', parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'])
customers   = pd.read_csv(f'{PROCESSED}/customers_clean.csv')
order_items = pd.read_csv(f'{PROCESSED}/order_items_clean.csv')
payments    = pd.read_csv(f'{PROCESSED}/payments_clean.csv')
products    = pd.read_csv(f'{PROCESSED}/products_clean.csv')
sellers     = pd.read_csv(f'{PROCESSED}/sellers_clean.csv')

print("All clean files loaded!")
print("Orders shape:", orders.shape)

All clean files loaded!
Orders shape: (96470, 8)


In [5]:
# Step 1: Join orders with customers
df = orders.merge(
    customers,
    on='customer_id',
    how='left'
)

print("After orders + customers:", df.shape)
print(df[['order_id', 'customer_id',
          'customer_city', 'customer_state']].head(3))

# Aggregate payments to one row per order
payments_agg = payments.groupby('order_id').agg(
    total_revenue = ('payment_value', 'sum'),
    payment_type  = ('payment_type', 'first'),
    installments  = ('payment_installments', 'max')
).reset_index()

print("Payments aggregated shape:", payments_agg.shape)
print(payments_agg.head(3))
df = df.merge(
    payments_agg,
    on='order_id',
    how='left'
)

print("After adding payments:", df.shape)


order_items_agg = order_items.groupby('order_id').agg(
    total_items   = ('order_item_id', 'count'),
    total_freight = ('freight_value', 'sum'),
    product_id    = ('product_id', 'first'),
    seller_id     = ('seller_id', 'first')
).reset_index()

print("Items aggregated shape:", order_items_agg.shape)

print("order_items aggregated shape:", order_items_agg.shape)
print(order_items_agg.head(3))
df = df.merge(
    order_items_agg,
    on='order_id',
    how='left'
)

# Select only columns we need from products
products_slim = products[['product_id',
                           'product_category_name_english']].copy()

df = df.merge(
    products_slim,
    on='product_id',
    how='left'
)

print("After adding products:", df.shape)

# Select only columns we need from sellers
sellers_slim = sellers[['seller_id',
                         'seller_state']].copy()

df = df.merge(
    sellers_slim,
    on='seller_id',
    how='left'
)

print("After adding sellers:", df.shape)

After orders + customers: (96470, 12)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  customer_city customer_state  
0     sao paulo             SP  
1     barreiras             BA  
2    vianopolis             GO  
Payments aggregated shape: (99440, 4)
                           order_id  total_revenue payment_type  installments
0  00010242fe8c5a6d1ba2dd792cb16214          72.19  credit_card             2
1  00018f77f2f0320c557190d7a144bdd3         259.83  credit_card             3
2  000229ec398224ef6ca0657da4fc703e         216.87  credit_card             5
After adding payments: (96470, 15)
Items aggregated shape: (98666, 5)
order_items aggregated shape: (98666, 5)
                           order_id  total_items  total_freight  \
0  000102

In [6]:
# 1. Delivery time in days
df['delivery_days'] = (
    df['order_delivered_customer_date'] -
    df['order_purchase_timestamp']
).dt.days

# 2. Extract month and year for trend analysis
df['order_month'] = df['order_purchase_timestamp'].dt.month
df['order_year']  = df['order_purchase_timestamp'].dt.year
df['order_month_year'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# 3. Was delivery on time?
df['delivered_on_time'] = (
    df['order_delivered_customer_date'] <=
    df['order_estimated_delivery_date']
)

print("New columns added!")
print(df[['delivery_days', 'order_month_year',
          'delivered_on_time']].head())

New columns added!
   delivery_days order_month_year  delivered_on_time
0              8          2017-10               True
1             13          2018-07               True
2              9          2018-08               True
3             13          2017-11               True
4              2          2018-02               True


In [7]:
# Final check
print("=== MASTER TABLE SUMMARY ===")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nSample:")
df.head(3)

=== MASTER TABLE SUMMARY ===
Shape: (96470, 26)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'total_revenue', 'payment_type', 'installments', 'total_items', 'total_freight', 'product_id', 'seller_id', 'product_category_name_english', 'seller_state', 'delivery_days', 'order_month', 'order_year', 'order_month_year', 'delivered_on_time']

Missing values:
order_approved_at                  14
order_delivered_carrier_date        1
total_revenue                       1
payment_type                        1
installments                        1
product_category_name_english    1378
dtype: int64

Sample:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,total_freight,product_id,seller_id,product_category_name_english,seller_state,delivery_days,order_month,order_year,order_month_year,delivered_on_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,8.72,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,housewares,SP,8,10,2017,2017-10,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,22.76,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,perfumery,SP,13,7,2018,2018-07,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,19.22,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,auto,SP,9,8,2018,2018-08,True


In [9]:
df['product_category_name_english'] = \
    df['product_category_name_english'].fillna('Unknown')

print("Nulls remaining:", df['product_category_name_english'].isnull().sum())

Nulls remaining: 0


In [8]:
df.to_csv(f'{PROCESSED}/master_orders.csv', index=False)
print("Master table saved!")
print("Final shape:", df.shape)

Master table saved!
Final shape: (96470, 26)
